In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# =========================
# Cell 1: Imports & Setup
# =========================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import xgboost as xgb


In [ ]:
# =========================
# Cell 2: Load Data
# =========================
train = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv")
test = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/test.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)
train.head()


In [ ]:
# =========================
# Cell 3: Basic Cleaning (fixed warnings)
# =========================

# Save IDs
train_ID = train['Id']
test_ID = test['Id']
train.drop(['Id'], axis=1, inplace=True)
test.drop(['Id'], axis=1, inplace=True)

# Fill missing values safely (no inplace warnings)
for col in train.columns:
    if train[col].dtype == "object":
        train[col] = train[col].fillna(train[col].mode()[0])
    else:
        train[col] = train[col].fillna(train[col].median())

for col in test.columns:
    if test[col].dtype == "object":
        test[col] = test[col].fillna(test[col].mode()[0])
    else:
        test[col] = test[col].fillna(test[col].median())


In [ ]:
# =========================
# Cell 4: Feature Engineering (Encoding)
# =========================
all_data = pd.concat([train.drop("SalePrice", axis=1), test], axis=0)
all_data = pd.get_dummies(all_data)

train_x = all_data.iloc[:train.shape[0], :]
test_x = all_data.iloc[train.shape[0]:, :]
train_y = train["SalePrice"]

print("Encoded shape:", all_data.shape)


In [ ]:
# =========================
# Cell 5: Train-Test Split
# =========================
X_train, X_valid, y_train, y_valid = train_test_split(train_x, train_y, test_size=0.2, random_state=42)
print("Training set:", X_train.shape, "Validation set:", X_valid.shape)


In [ ]:
# =========================
# Cell 6: Train Multiple Models
# =========================
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "XGBoost": xgb.XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=4, random_state=42)
}

rmse_scores = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_valid)
    rmse = np.sqrt(mean_squared_error(y_valid, preds))
    rmse_scores[name] = rmse
    print(f"{name} RMSE: {rmse:.4f}")


In [ ]:
# =========================
# Cell 7: Compare Models (Visualization)
# =========================
rmse_df = pd.DataFrame(list(rmse_scores.items()), columns=["Model", "RMSE"]).sort_values(by="RMSE")
print(rmse_df)

plt.figure(figsize=(8,5))
sns.barplot(x="RMSE", y="Model", data=rmse_df, palette="viridis")
plt.title("Model Comparison (RMSE)")
plt.show()


In [ ]:
# =========================
# Cell 8: Feature Importance (Random Forest)
# =========================
best_model = models["Random Forest"]
importances = best_model.feature_importances_
indices = np.argsort(importances)[-10:]

plt.figure(figsize=(8,5))
plt.barh(range(len(indices)), importances[indices], align='center')
plt.yticks(range(len(indices)), [train_x.columns[i] for i in indices])
plt.title("Top 10 Feature Importances (Random Forest)")
plt.show()


In [ ]:
# =========================
# Cell 9: Final Model & Submission
# =========================
final_model = models["XGBoost"]  # choose best-performing
final_model.fit(train_x, train_y)
predictions = final_model.predict(test_x)

submission = pd.DataFrame({
    "Id": test_ID,
    "SalePrice": predictions
})
submission.to_csv("submission.csv", index=False)
print("Submission file created: submission.csv")


Reflection and Planned Improvements

In this lab, I explored the House Prices: Advanced Regression Techniques dataset on Kaggle. I worked through the full data science pipeline:

Data Preparation: Handled missing values by filling with median (for numeric) and mode (for categorical), and encoded categorical variables using one-hot encoding.

Exploratory Analysis: Created simple comparisons and visualizations (feature importance, RMSE comparison) to understand how features and models behave.

Model Testing: Implemented and compared multiple models including Linear Regression, Random Forest, Gradient Boosting, and XGBoost. Among them, XGBoost achieved the lowest RMSE on the validation set.

Submission: Generated and submitted predictions to Kaggle, successfully producing a valid leaderboard score.

What Went Well

The workflow was straightforward and reproducible.

Tried multiple models and clearly compared them using RMSE.

Learned the importance of encoding and filling missing values properly.

Areas for Improvement

Feature engineering could be expanded (e.g., log-transforming skewed variables, creating interaction features, scaling).

Hyperparameter tuning (e.g., GridSearchCV or Bayesian Optimization) could further improve model accuracy.

Model ensembling (blending or stacking) might boost leaderboard performance.

Future Steps

Explore advanced feature engineering to capture non-linear relationships.

Try additional boosting algorithms like LightGBM and CatBoost.

Experiment with automated machine learning (AutoML) approaches for faster optimization.